# The Five R's of research software
Benureau and Rougier describe a hierarchy of requirements for research software. In general, any software should have the following properties:

* Re-runability
* Repeatability
* Reproducibility
* Reusability 
* Replicability

To demonstrate this they use a short piece of Python code designed to model a random walk of length 10 steps.


In [ ]:
import random

x=0
for i in xrange(10):
    step = random.choice([-1,+1])
    x += step
    print x,

# R1: Re-Runability
This code will not run in Python 3. The `xrange()` method and `print x,` syntax are only supported in Python 2. Unfortunately this is not evident in the script as no documentation (commenting) has been included. 

We can fix this by re-writing with Python3 syntax and/or making the python requirements explicit in the comments/docs.

In [11]:
# tested with Python 3
import random

x=0
walk = []
for i in range(10):
    step = random.choice([-1, +1])
    x += step
    walk.append(x)
    
print(walk)
    

[1, 2, 1, 2, 3, 2, 3, 4, 5, 4]


# R2: Repeatability
The code now runs but will not provide consistent results. This is a problem if our code is supposed to publish something we have published in an article. To address htis, we can set a "random seed". This seed will act as a starting point for `random` to start producing a list of predictable pseudo-random numbers. 

Be aware though! Randomness is much more complicated than it first seems. Parallel processing, for example, can really mess with predictable random processes. Take a look at `../notebooks/randomness_and_parallelism.ipynb` for more details.

For now, we'll set a random seed for our code using `ranodm.seed(1)`. We'll also add a line which writes the output todisk. This means we have a reliable reference for previous outputs.

In [12]:
# tested with Python 3
import random

random.seed(1) # RNG initilization
x=0
walk = []
for i in range(10):
    step = random.choice([-1, +1])
    x += step
    walk.append(x)
    
with open('R2_random_walk.txt', 'w') as f:
    f.write(str(walk))

## R3: Reproducibility
We now have code which will produce consistent results! Our next task is to put in some checks and balances to ensure that the code in the repository best reflects the methodlogy and results described in our research paper. 

To do this we're going to introduce a few different things:

* **Testing**: We're going to slightly modify the way our random walk works. Instead of using `random.choice()` we are now going to use the slightly more robost `random.uniform()`. To check that we haven't messed up our code in the process, we'll put the random walk code block in a function, and write a test which should have a known output. If this test fails, then the program will exit without wrtiing anything.
* **Version Control**: We're also going to use the `subprocess` library to allow Python to interact with the current repository and the command line. We will use this to check if the current repository is `clean`, i.e., if there are any un-committed changes to git. This is a great way to enforce version control on your projects, remembering that you can always revert to an older version if you need to - that's the point of version control, after all!
* **Metadata**: When the results are written to disk, we're also going to add extra information like the current git hash (most recent commit), date and time, random seed, and system information (Python version and OS).
* **Documentation**: We now have more information on the top rows, including copywrite and lisence info, and full python version (3.X.X) and operating system information. Rembember, especially when dealing with floating-point numbers, otherwise identical code can produce different results on a different OS. 

In [15]:
# Copyright (c) 2017 N.P Rougier and F.C.Y Berunreau
# Release under the BSD 2-clause license
# Tested with 64-bit CPython 3.6.2 / macOS 10.12.6

import sys, subprocess, random, datetime

def random_walk():
    x = 0
    walk = []
    for i in range(10):
        if random.uniform(-1,+1) < 0:
            x -= 1
        else:
            x += 1
        walk.append(x)
    return walk

## if repository is dirty don't run anything
# Check for uncommitted changes (staged & unstaged)
dirty = subprocess.call(['git', 'diff', '--quiet'])  # Returns 1 if dirty

# Check for untracked files
untracked = subprocess.check_output(['git', 'ls-files', '--others', '--exclude-standard']).strip()

# Check if we're on a valid branch
branch_status = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).strip().decode()

if dirty != 0 or untracked or branch_status == "HEAD":
    print('Repository dirty or on a detached HEAD state. Please commit and/or track your changes first.')
    sys.exit(1)
    
# check git hash (if any)
hash_cmd = ('git', 'rev-parse', 'HEAD')
revision = subprocess.check_output(hash_cmd)

# Unit test
random.seed(42)
assert random_walk() == [1, 0, -1, -2, -1, 0, 1, 0, -1, -2]

# random walk for 10 steps
seed = 1
random.seed(seed)
walk = random_walk()
print(walk)

# save results with metadata
results = {
    "data" : walk, 
    "seed" : seed,
    "timestamp" : str(datetime.datetime.utcnow()),
    "revision" : revision,
    "system" : sys.version    
}

with open('R3_random_walk.txt', 'w') as f:
    f.write(str(results))


[-1, 0, 1, 0, -1, -2, -1, 0, -1, -2]


## R4: Reusability
The code above does a great job of ensuring that our repostories are frequently updated and our code is tested, we've also begun to introduce functions, which can make code more reusable. The function isn't writen in a particularly reusable way, though. Imagine I want to use a random walk for another part of the project, so I try to import the method `random_walk()` into another peice of code. This will automatically force the whole of the rest of the script to run on import, potentially over-writing previous data. We also have no flexibility over any variables (x0, ranodm seed, walk length), making it difficult to use in other projects.

We can use the `if __name__ == '__main__'` syntax to get around this. Anything below this clause will not be run on import. Its good practice to have everything above this line secured inside functions, and then putting any function calling below this line.

We'll also add documentation to contextualise the functions and make them easier to use.

In [19]:
# Copyright (c) 2017 N.P Rougier and F.C.Y Berunreau
# Release under the BSD 2-clause license
# Tested with 64-bit CPython 3.6.2 / macOS 10.12.6
import sys, subprocess, random, datetime

def compute_walk(count, x0=0, step=1, seed=0):
    """Random walk.
    count (int) : number of steps.
    x0 (int) : start location.
    step (int) : step length.
    seed (int) : random seed.
    """
    random.seed(seed)
    x = x0
    walk = []
    for i in range(count):
        if random.uniform(-1,+1) < 0:
            x -= step
        else:
            x += step
        walk.append(x)
    return walk

def compute_results(count, x0=0, step=1, seed=0):
    """Compute random walk and return it woth context"""
    ## if repository is dirty don't run anything
    # Check for uncommitted changes (staged & unstaged)
    dirty = subprocess.call(['git', 'diff', '--quiet'])  # Returns 1 if dirty

    # Check for untracked files
    untracked = subprocess.check_output(['git', 'ls-files', '--others', '--exclude-standard']).strip()

    # Check if we're on a valid branch
    branch_status = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).strip().decode()

    if dirty != 0 or untracked or branch_status == "HEAD":
        print('Repository dirty or on a detached HEAD state. Please commit and/or track your changes first.')
        sys.exit(1)
        
    # check git hash (if any)
    hash_cmd = ('git', 'rev-parse', 'HEAD')
    revision = subprocess.check_output(hash_cmd)

    # random walk for 10 steps
    seed = 1
    random.seed(seed)
    walk = random_walk()
    print(walk)

    # save results with metadata
    results = {
        "data" : walk, 
        "seed" : seed,
        "timestamp" : str(datetime.datetime.utcnow()),
        "revision" : revision,
        "system" : sys.version    
    }
    return results

with open('random_walk.txt', 'w') as f:
    f.write(str(results))

if __name__ == "__main__":
    # Unit test (will fail with python <= 3.2)
    random.seed(42)
    assert random_walk() == [1, 0, -1, -2, -1, 0, 1, 0, -1, -2]
    
    # simulation parameters
    count, seed, step = 10, 1, 1
    results = compute_results(count=count, seed=seed, step=step)
    with open('R4_random_walk.txt', 'w') as f:
        f.write(str(results))

[-1, 0, 1, 0, -1, -2, -1, 0, -1, -2]


## R5: Replicability
Great, we've turned seven lines of code into 70! The new code may have taken 10x the time to write, but the steps we've taken will help keep your software clean, up-to-date and reusable. This will save you huge amounts of headaches in the future!

Replicability is a slightly more nebulous concept. Essentially, we want to make sure that the code is written well enough that someone could replicate it in, say, a different language, or using different or update packages. 

For example, lets say we want to compute a walk in parallel processing. The `random.seed()` method used to generate pre-determined random numbers will not work here. Instead, we are going to have to use the `numpy` package, which allows you to produce a randomg number generator object which can be passed between processors. We'll alter the code so that it uses this more robust form of randomisation. 

In [20]:
# Copyright (c) 2017 N.P Rougier and F.C.Y Berunreau
# Release under the BSD 2-clause license
# Tested with 64-bit CPython 3.6.2 / numpy 1.12.0 / macOS 10.12.6
import random
import numpy as np

def _rng(seed):
    """Returns a numpy random number generator initialised with a seed. For 
    consistency, this will be seeded as though it had used native python'set
    `random` package. 
    """
    rng = random.Random()
    rng.seed(seed)
    _, keys, _ = rng.getstate()
    rng = np.random.RandomState()
    state = rng.get_state()
    rng.set_state((state[0], keys[:-1], state[2], state[3], state[4]))
    return rng

def random_walk(n, seed):
    """Rndom walk for n steps"""
    rng = _rng(seed)
    steps = 2*(rng.uniform(-1,1, n)>0)-1
    return steps.cumsum().tolist()

if __name__=='__main__':
    # unit test
    assert (random_walk(n=10, seed=42) == [-1, -2, -1, -2, -3, -4, -3, -2, -1, -2])
    
    # random walk for 10 steps with seed 1
    seed=1
    walk = random_walk(n=10, seed=seed)
    
    # save results and seed
    results = {"seed":seed, "data":walk}
    
    with open('R5_random_walk.txt', 'w') as f:
        f.write(str(results))